In [1]:
from google.colab import drive
drive.mount("/content/drive")

MessageError: Error: credential propagation was unsuccessful

In [ ]:
!pip install transformers datasets accelerate --quiet

print("Libraries installed successfully")

Libraries installed successfully


In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import (
    DistilBertTokenizer,
    DistilBertForSequenceClassification,
    TrainingArguments,
    Trainer
)
from datasets import Dataset
from sklearn.metrics import accuracy_score, classification_report
import pickle

# Check GPU is available
print(f"GPU available: {torch.cuda.is_available()}")
print(f"GPU name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

GPU available: True
GPU name: NVIDIA A100-SXM4-80GB


In [ ]:
df = pd.read_csv("/content/drive/MyDrive/FactLens_Group9/data/df_cleaned.csv")

print(f"Dataset loaded: {len(df)} articles")
print(df["label"].value_counts())

Dataset loaded: 38590 articles
label
REAL    21191
FAKE    17399
Name: count, dtype: int64


In [ ]:
# Convert labels to numbers - DistilBERT needs integers
# FAKE = 0, REAL = 1
df["numeric_label"] = df["label"].map({"FAKE": 0, "REAL": 1})

print("Label mapping:")
print("  FAKE → 0")
print("  REAL → 1")
print(f"\nLabel distribution:")
print(df["numeric_label"].value_counts())

Label mapping:
  FAKE → 0
  REAL → 1

Label distribution:
numeric_label
1    21191
0    17399
Name: count, dtype: int64


In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df[["cleaned_text", "numeric_label"]],
    test_size=0.2,
    random_state=42,
    stratify=df["numeric_label"]
)

print(f"Training set: {len(train_df)} articles")
print(f"Test set:     {len(test_df)} articles")
print(f"\nTraining label distribution:")
print(train_df["numeric_label"].value_counts())

Training set: 30872 articles
Test set:     7718 articles

Training label distribution:
numeric_label
1    16953
0    13919
Name: count, dtype: int64


In [ ]:
print("Loading DistilBERT tokenizer...")

tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

print("Tokenizer loaded successfully")
print(f"Vocabulary size: {tokenizer.vocab_size:,}")

Loading DistilBERT tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer loaded successfully
Vocabulary size: 30,522


In [ ]:
def tokenize(batch):
    return tokenizer(
        batch["cleaned_text"],
        padding=True,
        truncation=True,
        max_length=512
    )

# Convert to Hugging Face datasets
train_dataset = Dataset.from_pandas(train_df.rename(
    columns={"numeric_label": "labels"}
).reset_index(drop=True))

test_dataset = Dataset.from_pandas(test_df.rename(
    columns={"numeric_label": "labels"}
).reset_index(drop=True))

# Apply tokenization
print("Tokenizing training set...")
train_dataset = train_dataset.map(tokenize, batched=True, batch_size=512)

print("Tokenizing test set...")
test_dataset = test_dataset.map(tokenize, batched=True, batch_size=512)

# Set format for PyTorch
train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

print(f"\nTokenization complete")
print(f"Training dataset: {len(train_dataset)} articles")
print(f"Test dataset:     {len(test_dataset)} articles")

Tokenizing training set...


Map:   0%|          | 0/30872 [00:00<?, ? examples/s]

Tokenizing test set...


Map:   0%|          | 0/7718 [00:00<?, ? examples/s]


Tokenization complete
Training dataset: 30872 articles
Test dataset:     7718 articles


In [ ]:
print("Loading DistilBERT model...")

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

print("Model loaded successfully")
print(f"Number of parameters: {sum(p.numel() for p in model.parameters()):,}")

Loading DistilBERT model...


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded successfully
Number of parameters: 66,955,010


In [ ]:
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/FactLens_Group9/data/distilbert_checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir="/content/logs",
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    fp16=True,
    report_to="none"
)

print("Training arguments set successfully")
print(f"\nKey settings:")
print(f"  Epochs:     3")
print(f"  Batch size: 32")
print(f"  FP16:       True (faster training on GPU)")

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Training arguments set successfully

Key settings:
  Epochs:     3
  Batch size: 32
  FP16:       True (faster training on GPU)


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    accuracy = accuracy_score(labels, predictions)
    return {"accuracy": accuracy}

print("Metrics function defined successfully")

Metrics function defined successfully


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

print("Starting training...")
print("This will take 15-25 minutes on A100 GPU...")
print("You will see progress after each epoch\n")

trainer.train()

print("\nTraining complete")

Starting training...
This will take 15-25 minutes on A100 GPU...
You will see progress after each epoch



Epoch,Training Loss,Validation Loss,Accuracy
1,0.018273,0.014426,0.997150
2,0.004037,0.002406,0.999223
3,0.000143,0.001583,0.999741


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].



Training complete


In [ ]:
print("Evaluating on test set...")

predictions = trainer.predict(test_dataset)
y_pred = np.argmax(predictions.predictions, axis=1)
y_true = test_df["numeric_label"].values

accuracy = accuracy_score(y_true, y_pred)

print(f"\nDistilBERT Test Accuracy: {accuracy:.4f} ({accuracy:.2%})")
print("\nDetailed Classification Report:")
print(classification_report(y_true, y_pred,
      target_names=["FAKE", "REAL"]))

Evaluating on test set...



DistilBERT Test Accuracy: 0.9997 (99.97%)

Detailed Classification Report:
              precision    recall  f1-score   support

        FAKE       1.00      1.00      1.00      3480
        REAL       1.00      1.00      1.00      4238

    accuracy                           1.00      7718
   macro avg       1.00      1.00      1.00      7718
weighted avg       1.00      1.00      1.00      7718



In [ ]:
print("=" * 60)
print("FULL MODEL COMPARISON")
print("=" * 60)
print(f"Original LR — TF-IDF only:          98.52%")
print(f"Debiased LR — no source words:      97.50%")
print(f"Enhanced LR — TF-IDF + features:    98.33%")
print(f"DistilBERT — fine-tuned:            {accuracy:.2%}")
print("=" * 60)

FULL MODEL COMPARISON
Original LR — TF-IDF only:          98.52%
Debiased LR — no source words:      97.50%
Enhanced LR — TF-IDF + features:    98.33%
DistilBERT — fine-tuned:            99.97%


In [ ]:
# Save model and tokenizer to Drive
save_path = "/content/drive/MyDrive/FactLens_Group9/data/distilbert_model"

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

# Save predictions for LIME/SHAP in Step 10
np.save("/content/drive/MyDrive/FactLens_Group9/data/distilbert_predictions.npy",
        y_pred)
np.save("/content/drive/MyDrive/FactLens_Group9/data/distilbert_true_labels.npy",
        y_true)

print(f"Model saved to {save_path}")
print("Predictions saved successfully")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to /content/drive/MyDrive/FactLens_Group9/data/distilbert_model
Predictions saved successfully
